# Session 6: Cell segmentation and quality control

Almost every spatial omics analysis starts by drawing an outline around each cell. Everything after that
depends on those outlines: a transcript is counted for a cell only if it falls inside the cell's outline, and
a protein intensity is averaged over the pixels inside it. An outline that is too small loses transcripts; one
that spills into a neighbour picks up the neighbour's transcripts, and can make a T cell appear to express a
tumour gene.

10x's software has already segmented this section, and its outlines are in the workshop data, together with
the transcripts and the H&E image. In this session you will:

1. meet **SpatialData**, the container that holds the image, transcripts, outlines and tables together;
2. look at how 10x segmented the cells;
3. segment the same cells yourself with **InstanSeg**, the model you used in QuPath on Day 1, run through
   **Sopa**;
4. turn the outlines into a cell-by-gene table (**aggregation**);
5. compare the segmentations with **SegTraQ**, a toolkit of segmentation quality metrics, including the cells you
   found in session 5;
6. change the segmentation and see how the metrics respond.

There is no ground truth here: nobody has drawn the "correct" outlines by hand. The aim is to learn which
questions to ask of a segmentation, and how to answer them with numbers rather than by eye alone.

Cells marked **Exercise** are for you to complete. The completed notebook is in `solutions/`.

## Setup

The workshop data should be in the `data` folder of the workshop folder (see the installation instructions).
If you put it somewhere else, change `DATA` below.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import segtraq
import sopa
import spatialdata as sd
from spatialdata.models import PointsModel, ShapesModel
from spatialdata.transformations import Identity, Scale, get_transformation

segtraq.settings.progress = False  # SegTraQ's progress bars add little in a notebook

DATA = Path("../data/xenium_rcc_crop.zarr")

# False: work on a 440 µm region in the middle of the crop, where every step takes less than a minute.
# True: run the whole session on the whole 1.5 mm crop instead. This takes much longer (see section 2).
WHOLE_CROP = False

sdata = sd.read_zarr(DATA)
PIXEL_SIZE = sdata.attrs["pixel_size_um"]  # µm per image pixel: 0.2125 for Xenium
sdata

## 1. One container for everything: SpatialData

In session 5 you worked with separate files: an OME-TIFF for the image and GeoJSON for the outlines. That works
for images and objects, but spatial omics adds much more: about two million transcripts, each with a position and
a gene name; tables of measurements per cell; and an H&E image of the same section, scanned on another microscope
at another pixel size.

[SpatialData](https://spatialdata.scverse.org) keeps all of these together, in one folder on disk (a **Zarr**
store), as **elements** of five kinds:

| Element | What it holds | Here | In session 5 |
|---|---|---|---|
| Images | pixels, as a pyramid of resolutions | `morphology_focus` (35 channels), `he` | the OME-TIFF |
| Points | single coordinates with properties | `transcripts` | (none) |
| Shapes | outlines (polygons) or circles | `cell_boundaries`, `nucleus_boundaries`, `pathology_annotations` | GeoJSON |
| Tables | one row per cell, as an AnnData object | `table` (genes), `protein` | the measurement table |
| Labels | label images (one number per object) | (none) | the label images you made |

The printout above lists the elements. Nothing has been read into memory yet: each element is **lazy**, like the
image in session 5, and pixels or transcripts are read only when you ask for them.

In [ ]:
image = sdata["morphology_focus"]
for level in image.children:  # the pyramid: each level halves the resolution
    print(level, dict(image[level].image.sizes))
image["scale0"].image.data  # a lazy dask array: 35 channels, about 3.5 GB if read in full

### Transcripts

Each transcript is a point with an `x`, `y` and `z` position (in µm), the gene it detects (`feature_name`), the
10x cell it was assigned to (`cell_id`), and a quality score (`qv`). Counting them reads only the columns needed.

In [ ]:
points = sdata["transcripts"]
print(f"{len(points):,} transcripts of {points['feature_name'].nunique().compute()} genes")
print("Most frequent genes:")
print(points["feature_name"].value_counts().compute().sort_values(ascending=False).head(8))
points.head()

### Tables

Tables are [AnnData](https://anndata.readthedocs.io) objects, the format scanpy uses: a matrix `X` with one row per
cell (`obs`) and one column per feature (`var`). The workshop data has two tables over the same cells:

- `table`: transcript counts for the 405 genes;
- `protein`: the mean intensity of the 27 antibody channels in each cell.

They are kept apart because counts and intensities need different normalisation (session 7). Each table says which
outlines it describes, and which column links its rows to them:

In [ ]:
print(sdata["table"].uns["spatialdata_attrs"])
sdata["table"].obs[["cell_id", "transcript_counts", "cell_area", "nucleus_area"]].head()

That link is what lets you draw a table's values onto the outlines. Here are 10x's cells in a 200 µm window,
coloured by their CD8A protein intensity:

In [ ]:
WINDOW = [940, 840, 1140, 1040]  # x0, y0, x1, y1 in µm: the tile you analysed in session 5

protein = sdata["protein"]
cd8a = pd.Series(protein[:, "CD8A"].X.toarray().ravel(), index=protein.obs["cell_id"].astype(str).values)

cells = sdata["cell_boundaries"]  # stored in µm
cells = cells.cx[WINDOW[0]:WINDOW[2], WINDOW[1]:WINDOW[3]]
ax = cells.assign(CD8A=cd8a.reindex(cells.index).values).plot(column="CD8A", cmap="viridis", legend=True,
                                                               figsize=(7, 6))
ax.set(xlim=(WINDOW[0], WINDOW[2]), ylim=(WINDOW[3], WINDOW[1]), xlabel="x (µm)", ylabel="y (µm)",
       title="10x cells, coloured by CD8A protein")
plt.show()

### Coordinate systems

Different elements are stored in different units: the image in pixels, the transcripts and 10x's outlines in µm,
the H&E in its own scanner's pixels. Each element carries **transformations** that say how to place it in each
**coordinate system**. The workshop data has two: `global`, in image pixels (as in QuPath), and `microns`.

In [ ]:
for name in ["morphology_focus", "transcripts", "cell_boundaries", "he"]:
    print(f"{name}:")
    for system, transformation in get_transformation(sdata[name], get_all=True).items():
        print(f"  {system}: {transformation}")

The image and the transcripts need only a scale (0.2125 µm per pixel). The H&E needs more: it was scanned after
the Xenium run, at about 0.27 µm per pixel and rotated by 90°, so its transformation is an **affine** one
(rotation, scale and shift together). 10x worked it out by matching the H&E to the DAPI image.

SpatialData applies the transformation when you ask for an element in a coordinate system. `rasterize` turns a
window of any image into pixels of a chosen size in that system, so the H&E and the immunofluorescence line up:

In [ ]:
def window_image(image_key, window, channels=None):
    """A window of an image in µm, resampled to the Xenium pixel size."""
    data = sd.bounding_box_query(sdata[image_key], axes=("x", "y"), min_coordinate=window[:2],
                                 max_coordinate=window[2:], target_coordinate_system="microns")
    pixels = sd.rasterize(data, axes=("x", "y"), min_coordinate=window[:2], max_coordinate=window[2:],
                          target_coordinate_system="microns", target_unit_to_pixels=1 / PIXEL_SIZE)
    return pixels if channels is None else pixels.sel(c=channels)


SMALL = [1000, 900, 1080, 980]  # an 80 µm window
he = window_image("he", SMALL).transpose("y", "x", "c").values
dapi = window_image("morphology_focus", SMALL, "DAPI").values
cells = sdata["cell_boundaries"].cx[SMALL[0]:SMALL[2], SMALL[1]:SMALL[3]]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(he, extent=[SMALL[0], SMALL[2], SMALL[3], SMALL[1]])
axes[0].set_title("H&E")
axes[1].imshow(dapi, cmap="gray", vmax=np.percentile(dapi, 99.5), extent=[SMALL[0], SMALL[2], SMALL[3], SMALL[1]])
axes[1].set_title("DAPI")
for ax in axes:
    cells.boundary.plot(ax=ax, color="yellow", linewidth=0.8)
    ax.set(xlim=(SMALL[0], SMALL[2]), ylim=(SMALL[3], SMALL[1]), xlabel="x (µm)", ylabel="y (µm)")
plt.show()

The same yellow outlines (10x's cells) sit on the same nuclei in both images.

### Bringing in objects from QuPath

QuPath exports GeoJSON in pixels of the image it was drawn on. The workshop's Day 1 images start at the same corner
and have the same pixel size as `morphology_focus`, so objects from QuPath belong in the `global` coordinate system
as they are, and in `microns` after a scale. The function below reads a QuPath GeoJSON file into a SpatialData
shapes element with those two transformations, keeping each object's classification.

We use it on the cells you saved at the end of session 5 (`session5_cells.geojson`), which we will compare with the
other segmentations in section 7. You could load your Day 1 annotations the same way.

In [ ]:
def read_qupath_geojson(path):
    """Read QuPath GeoJSON (in image pixels) as a SpatialData shapes element."""
    objects = gpd.read_file(path).set_crs(None, allow_override=True)  # pixels, not longitude/latitude
    shapes = gpd.GeoDataFrame(geometry=objects.geometry.values)
    if "classification" in objects:
        shapes["classification"] = [c["name"] if isinstance(c, dict) else None for c in objects["classification"]]
    shapes.index = [f"qupath-{i + 1}" for i in range(len(shapes))]
    return ShapesModel.parse(
        shapes, transformations={"global": Identity(), "microns": Scale([PIXEL_SIZE, PIXEL_SIZE], axes=("x", "y"))}
    )


SESSION5_CELLS = DATA.parent / "session5_cells.geojson"
if SESSION5_CELLS.exists():
    session5_cells = read_qupath_geojson(SESSION5_CELLS)
    print(f"{len(session5_cells)} cells from session 5")
    print(session5_cells["classification"].value_counts())
else:
    session5_cells = None
    print(f"{SESSION5_CELLS} not found: run the last section of session 5 to create it. Section 7 will be skipped.")

## 2. Choose a region to work on

The workshop crop is 1.5 × 1.5 mm and holds about 16,000 cells. Segmenting all of it with InstanSeg takes about
6 minutes on a fast desktop computer without a GPU, and longer on most laptops, and this session segments it
three times. So by default we work on a smaller square, 440 µm across, with about 1,300 cells, centred on the tile
you analysed in session 5. Every step takes less than a minute on it.

To run the whole session on the whole crop instead, set `WHOLE_CROP = True` in the setup cell and run the
notebook again, perhaps over lunch. Every step below works the same way on either.

Regions are given in µm, using the Zarr's `microns` coordinate system.

In [ ]:
REGION = [820, 720, 1260, 1160]  # x0, y0, x1, y1 in µm: around the session 5 tile

if WHOLE_CROP:  # the whole crop, from the size of the image
    full = sdata["morphology_focus"]["scale0"].image
    REGION = [0, 0, full.sizes["x"] * PIXEL_SIZE, full.sizes["y"] * PIXEL_SIZE]
print(f"Working on x {REGION[0]:.0f}–{REGION[2]:.0f} µm, y {REGION[1]:.0f}–{REGION[3]:.0f} µm")

# A low-resolution overview of DAPI, with the region and the pathologist's annotations
overview = sdata["morphology_focus"]["scale3"].image.sel(c="DAPI")
factor = sdata["morphology_focus"]["scale0"].image.sizes["x"] / overview.sizes["x"]
extent_um = [0, overview.sizes["x"] * factor * PIXEL_SIZE, overview.sizes["y"] * factor * PIXEL_SIZE, 0]

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(overview, cmap="gray", vmax=np.percentile(overview, 99.5), extent=extent_um)
annotations = sd.transform(sdata["pathology_annotations"], to_coordinate_system="microns")
annotations.boundary.plot(ax=ax, color="yellow", linewidth=1)
for _, row in annotations.iterrows():
    point = row.geometry.representative_point()
    ax.annotate(row["name"], (point.x, point.y), color="yellow", ha="center")
ax.add_patch(plt.Rectangle(REGION[:2], REGION[2] - REGION[0], REGION[3] - REGION[1], fill=False, color="red", lw=2))
ax.set(title="DAPI, pathology annotations (yellow) and the region (red)", xlabel="x (µm)", ylabel="y (µm)")
plt.show()

`bounding_box_query` cuts every element (images, transcripts, outlines and tables) to the region in one go. We
keep only what this session needs in a new `SpatialData` object, called `region`.

A new object has a second benefit. `sdata` was read from the workshop Zarr, and Sopa saves its results into
the Zarr of the object it works on. Working on `region`, which lives only in memory, leaves the workshop data
untouched.

In [ ]:
query = sd.bounding_box_query(
    sdata, axes=("x", "y"), min_coordinate=REGION[:2], max_coordinate=REGION[2:], target_coordinate_system="microns"
)
region = sd.SpatialData(
    images={"morphology_focus": query["morphology_focus"]},
    points={"transcripts": query["transcripts"]},
    shapes={"cell_boundaries": query["cell_boundaries"], "nucleus_boundaries": query["nucleus_boundaries"]},
    tables={"table": query["table"]},
)
region

## 3. How did 10x segment these cells?

10x's segmentation is recorded in `table.obs`. Xenium segments in steps: it finds nuclei in the DAPI image,
then finds each cell's edge from a **boundary stain** (here a mix of antibodies against ATP1A1, CD45 and
E-cadherin, which outline most cell types). Where the boundary stain is too weak, it falls back to growing the
nucleus outward by a fixed distance, or to the interior stain.

In [ ]:
region["table"].obs["segmentation_method"].value_counts()

To compare outlines we need pictures. The helper below draws a small window of the image in µm, DAPI in blue
and the boundary stain in magenta, with any number of outline sets on top. Outlines from different sources
can be stored in different units (10x's in µm, InstanSeg's in pixels), so `outlines()` first converts
them to µm with `sd.transform`.

In [ ]:
ZOOM = (1000, 900, 1100, 1000)  # a 100 µm window inside the session 5 tile: x0, y0, x1, y1 in µm
BOUNDARY_STAIN = "ATP1A1/CD45/E-Cadherin"


def outlines(key, data=None):
    """The outlines `key` from `data` (by default `region`), in µm."""
    data = region if data is None else data
    return sd.transform(data[key], to_coordinate_system="microns")


def stretch(channel):
    """Scale an image channel to 0–1 for display, saturating the brightest 0.5% of pixels."""
    low, high = np.percentile(channel, [1, 99.5])
    return np.clip((channel - low) / (high - low + 1e-9), 0, 1)


def show(window, layers, ax=None, title=None):
    """Draw DAPI (blue) and the boundary stain (magenta) in `window`, with outlines on top.

    `layers` maps a label to (outlines in µm, colour)."""
    x0, y0, x1, y1 = window
    pixels = slice(int(y0 / PIXEL_SIZE), int(y1 / PIXEL_SIZE)), slice(int(x0 / PIXEL_SIZE), int(x1 / PIXEL_SIZE))
    image = sdata["morphology_focus"]["scale0"].image
    dapi = stretch(image.sel(c="DAPI").values[pixels])
    boundary = stretch(image.sel(c=BOUNDARY_STAIN).values[pixels])
    rgb = np.stack([boundary, np.zeros_like(dapi), np.maximum(dapi, boundary)], axis=-1)

    if ax is None:
        _, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(rgb, extent=[x0, x1, y1, y0])
    for label, (shapes, colour) in layers.items():
        shapes.cx[x0:x1, y0:y1].boundary.plot(ax=ax, color=colour, linewidth=1, label=label)
    ax.set(xlim=(x0, x1), ylim=(y1, y0), xlabel="x (µm)", ylabel="y (µm)", title=title)
    ax.legend(loc="lower right", fontsize="small")
    return ax


show(ZOOM, {"10x cells": (outlines("cell_boundaries"), "yellow"), "10x nuclei": (outlines("nucleus_boundaries"), "cyan")},
     title="10x segmentation")
plt.show()

**Questions**

- Do the yellow cell outlines follow the magenta boundary stain? Where don't they?
- Are there cells with no nucleus, or nuclei with no cell?
- 10x's outlines tile the tissue with almost no gaps. Is that realistic for this tissue?

## 4. Segment the cells with InstanSeg

Sopa runs segmentation on large images in three steps:

1. **patches:** split the image into overlapping squares (`make_image_patches`), small enough to process one at a
   time;
2. **segment** each patch (here with InstanSeg);
3. **resolve:** merge the outlines from all patches, removing duplicates where patches overlap.

Steps 2 and 3 both happen inside `sopa.segmentation.instanseg`. On a whole slide, the patches can be spread over
many computers; here they run one after the other.

InstanSeg's `fluorescence_nuclei_and_cells` model was trained on images at 0.5 µm per pixel, so we tell it our
pixel size and it rescales the image to match. **Without `pixel_size` it can return no cells at all.** The
model accepts any number of channels, in any order, which is unusual for a segmentation model. We start with
DAPI alone.

The first run downloads the model, if `pixi run check-full` hasn't already done so.

In [ ]:
sopa.make_image_patches(region, patch_width=1024, patch_overlap=50, image_key="morphology_focus")

sopa.segmentation.instanseg(
    region, channels=["DAPI"], pixel_size=PIXEL_SIZE, image_key="morphology_focus", key_added="instanseg_dapi"
)

**Exercise 1.** Run InstanSeg again with two channels, DAPI and the boundary stain (`BOUNDARY_STAIN`), and save the
outlines as `"instanseg_cells"`.

In [ ]:
sopa.segmentation.instanseg(
    region,
    channels=["DAPI", BOUNDARY_STAIN],
    pixel_size=PIXEL_SIZE,
    image_key="morphology_focus",
    key_added="instanseg_cells",
)

Let's compare the three segmentations: the number of cells and their sizes, in µm².

In [ ]:
segmentations = {
    "10x": outlines("cell_boundaries"),
    "InstanSeg, DAPI": outlines("instanseg_dapi"),
    "InstanSeg, DAPI + boundary": outlines("instanseg_cells"),
}
pd.DataFrame(
    {name: {"cells": len(shapes), "median area (µm²)": shapes.area.median()} for name, shapes in segmentations.items()}
).T.round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (name, shapes) in zip(axes, segmentations.items()):
    show(ZOOM, {name: (shapes, "yellow")}, ax=ax, title=name)
plt.show()

**Questions**

- What did adding the boundary stain change?
- InstanSeg's outlines leave gaps between cells, and 10x's don't. Which is closer to what you see in the image?
- Where do the segmentations disagree most? Is that where the image is hard to read by eye too?

For quality control later we also need InstanSeg's **nuclei**. The same model outputs nuclei and cells; `target`
chooses which.

In [ ]:
sopa.segmentation.instanseg(
    region,
    channels=["DAPI", BOUNDARY_STAIN],
    pixel_size=PIXEL_SIZE,
    image_key="morphology_focus",
    target="nuclei",
    key_added="instanseg_nuclei",
)

## 5. From outlines to a cell table: aggregation

Outlines on their own aren't data you can cluster. **Aggregation** turns them into a table with one row per cell:

- each transcript is counted for the cell it falls inside, giving a cell-by-gene count matrix (`X`);
- each image channel is averaged over the cell's pixels, giving the protein intensities (`obsm["intensities"]`).

Two settings matter here:

- `key_added`: by default Sopa saves the new table as `"table"`, which would **overwrite 10x's table**. We name
  ours `"instanseg_table"`.
- `min_transcripts` and `min_intensity_ratio` drop cells with few transcripts or little signal. We set both to 0
  to keep every cell, so that the quality metrics below see the segmentation as it is.

In [ ]:
sopa.aggregate(
    region,
    shapes_key="instanseg_cells",
    image_key="morphology_focus",
    points_key="transcripts",
    gene_column="feature_name",
    key_added="instanseg_table",
    min_transcripts=0,
    min_intensity_ratio=0,
)
region["instanseg_table"]

`X` holds the transcript counts, and `obsm["intensities"]` the mean intensity of each of the 35 image channels.

Note that Sopa names its cells in the same style as 10x (`aaaaaaaa-1`, `aaaaaaab-1`, ...). The names only mean
something within one segmentation: InstanSeg's cell `aaaaaaab-1` has nothing to do with any 10x cell.

In [ ]:
region["instanseg_table"].obsm["intensities"].iloc[:5, :8]

Each transcript in `region["transcripts"]` carries the `cell_id` of the 10x cell it was assigned to
(`UNASSIGNED` if none). SegTraQ needs the same for InstanSeg, so we add a column with the InstanSeg cell each
transcript falls in. Sopa numbers the cells from 1 and uses 0 for "outside every cell"; we convert the numbers
to the cell names used in the table.

In [ ]:
sopa.spatial.assign_transcript_to_cell(
    region, points_key="transcripts", shapes_key="instanseg_cells", key_added="instanseg_cell", unassigned_value=0
)


def cell_names(numbers, shapes_key):
    """Convert Sopa's cell numbers (1, 2, ...; 0 = none) to the cell names of `shapes_key`."""
    names = np.asarray(region[shapes_key].index)
    numbers = np.asarray(numbers)
    return np.where(numbers == 0, "UNASSIGNED", names[np.maximum(numbers - 1, 0)])


transcripts = region["transcripts"].compute()
transcripts["instanseg_cell_id"] = cell_names(transcripts["instanseg_cell"], "instanseg_cells")
transcripts[["feature_name", "cell_id", "instanseg_cell_id"]].head()

## 6. Quality control with SegTraQ

[SegTraQ](https://github.com/LazDaria/segtraq) computes many segmentation quality metrics in a few families. We
use two:

- **Baseline** (`bl`): how many cells, how many transcripts fall outside every cell, transcripts and genes per
  cell, and cell shape.
- **Region similarity** (`rs`): how each cell relates to its nucleus, and whether the transcripts at a cell's
  edge look like the cell's own or like its neighbours' (the **border admixture score**).

SegTraQ looks at one segmentation at a time, and expects its transcripts, outlines and table under fixed names,
all in the same units. The helper below packages one segmentation that way, in µm. It also lets SegTraQ compute
cell areas and centroids from the outlines, so both segmentations are measured the same way.

In [ ]:
def run_segtraq(transcript_cell_ids, cells, nuclei, table, points=None):
    """Run SegTraQ's baseline and region-similarity metrics on one segmentation.

    transcript_cell_ids: the cell each transcript is assigned to ("UNASSIGNED" if none)
    cells, nuclei: the cell and nucleus outlines, in µm
    table: the segmentation's cell-by-gene table
    points: the transcripts to use (by default, all of them in the region)
    """
    points = (transcripts if points is None else points)[["x", "y", "z", "feature_name", "qv"]].copy()
    points["cell_id"] = np.asarray(transcript_cell_ids)

    cells = cells.rename_axis("cell_id")
    cells.attrs, nuclei = {}, nuclei.reset_index(drop=True)
    nuclei.attrs = {}

    table = table.copy()
    table.obs = pd.DataFrame({"cell_id": table.obs["cell_id"].astype(str).values,
                              "region": pd.Categorical(["cell_boundaries"] * table.n_obs)}, index=table.obs_names)
    table.uns["spatialdata_attrs"] = {"region": "cell_boundaries", "region_key": "region", "instance_key": "cell_id"}

    package = sd.SpatialData(
        points={"transcripts": PointsModel.parse(points, transformations={"microns": Identity()})},
        shapes={
            "cell_boundaries": ShapesModel.parse(cells, transformations={"microns": Identity()}),
            "nucleus_boundaries": ShapesModel.parse(nuclei, transformations={"microns": Identity()}),
        },
        tables={"table": table},
    )
    qc = segtraq.SegTraQ(
        package,
        images_key=None,
        tables_area_key=None,  # compute areas and centroids from the outlines
        tables_centroid_x_key=None,
        tables_centroid_y_key=None,
        nucleus_shapes_cell_id_key=None,  # nuclei are matched to cells by overlap, not by name
    )
    return qc.run_baseline(inplace=False) | qc.run_region_similarity(inplace=False)


qc = {
    "10x": run_segtraq(transcripts["cell_id"], outlines("cell_boundaries"), outlines("nucleus_boundaries"),
                       region["table"]),
    "InstanSeg": run_segtraq(transcripts["instanseg_cell_id"], outlines("instanseg_cells"),
                             outlines("instanseg_nuclei"), region["instanseg_table"]),
}

SegTraQ returns a dictionary of results per segmentation. Some are single numbers, and most are tables with one
row per cell. Let's put the headline numbers side by side.

SegTraQ first drops control probes and low-quality transcripts (quality score below 20), so its transcript
counts are a little lower than the raw ones.

In [ ]:
def summary(results):
    return {
        "cells": results["num_cells"],
        "transcripts outside every cell (%)": results["perc_unassigned_transcripts"],
        "median transcripts per cell": results["transcripts_per_cell"]["transcript_count"].median(),
        "median genes per cell": results["genes_per_cell"]["gene_count"].median(),
        "median cell area (µm²)": results["morphological_features"]["cell_area"].median(),
        "median nucleus–cell IoU": results["ious"]["iou"].median(),
        "median border admixture": results["border_admixture_score"]["border_admixture_score"].median(),
    }


pd.DataFrame({name: summary(results) for name, results in qc.items()}).round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
panels = [
    ("transcripts_per_cell", "transcript_count", "transcripts per cell", np.arange(0, 400, 10)),
    ("morphological_features", "cell_area", "cell area (µm²)", np.arange(0, 300, 5)),
    ("border_admixture_score", "border_admixture_score", "border admixture score", np.linspace(-0.4, 0.8, 49)),
]
for ax, (key, column, label, bins) in zip(axes, panels):
    for name, results in qc.items():
        ax.hist(results[key][column].dropna(), bins=bins, alpha=0.5, label=name)
    ax.set(xlabel=label, ylabel="cells")
axes[0].legend()
plt.show()

**What the metrics mean**

- **Transcripts outside every cell.** Some transcripts really are outside cells, but a high value usually means
  the outlines are too tight and are losing transcripts that belong to cells.
- **Transcripts and genes per cell.** More is better only up to a point: oversized outlines also gain transcripts,
  from their neighbours.
- **Nucleus–cell IoU** (intersection over union) compares each cell with its nucleus: 1 means the cell is no
  bigger than its nucleus, and small values mean a lot of cytoplasm, or an oversized outline.
- **Border admixture** compares the transcripts near a cell's edge with those in its centre and those of its
  neighbours. Higher values mean the edge looks more like the neighbours, which suggests the outline has crossed
  into them.

**Questions**

- InstanSeg finds a similar number of cells to 10x. Why do so many more transcripts fall outside its cells?
- Which segmentation would you trust more for asking "which cells express CD8A"? And for "how many cells are in
  this region"?
- Is either segmentation clearly better on every metric?

## 7. Your session 5 cells

At the end of session 5 you saved the cells you found with a threshold, a watershed and a fixed expansion. How do
they compare with InstanSeg's and 10x's? They exist only in the session 5 tile, so we compare all three there.

We add them to `region`, aggregate and assign transcripts exactly as for InstanSeg, then run SegTraQ on the tile.
Cells are counted as in the tile if their centre is in it; transcripts assigned to cells whose centre lies outside
are left out, so that cells cut by the tile's edge don't count against any segmentation. For nuclei we use
InstanSeg's for both InstanSeg and the session 5 cells, which didn't keep their nuclei.

In [ ]:
TILE = [940, 840, 1140, 1040]  # the session 5 tile, x0, y0, x1, y1 in µm


def in_tile(shapes):
    """The outlines (in µm) whose centre lies in the tile."""
    centres = shapes.centroid
    return shapes[(centres.x >= TILE[0]) & (centres.x < TILE[2]) & (centres.y >= TILE[1]) & (centres.y < TILE[3])]


def tile_qc(cell_ids, cells, nuclei, table):
    """SegTraQ on the tile only, for one segmentation (cell_ids: one per transcript in tile_points)."""
    cells = in_tile(cells)
    cell_ids = pd.Series(np.asarray(cell_ids).astype(str), index=tile_points.index)
    keep = cell_ids.isin(cells.index.astype(str)) | (cell_ids == "UNASSIGNED")
    table = table[table.obs["cell_id"].astype(str).isin(cells.index.astype(str))]
    return run_segtraq(cell_ids[keep], cells, in_tile(nuclei), table, points=tile_points[keep])


if session5_cells is None:
    print("No session 5 cells: skipping this section.")
else:
    region["session5_cells"] = session5_cells
    sopa.aggregate(region, shapes_key="session5_cells", image_key="morphology_focus", points_key="transcripts",
                   gene_column="feature_name", key_added="session5_table", min_transcripts=0, min_intensity_ratio=0)
    sopa.spatial.assign_transcript_to_cell(region, points_key="transcripts", shapes_key="session5_cells",
                                           key_added="session5_cell", unassigned_value=0)

    points = region["transcripts"].compute()
    tile_points = points[(points["x"] >= TILE[0]) & (points["x"] < TILE[2])
                         & (points["y"] >= TILE[1]) & (points["y"] < TILE[3])]
    qc_tile = {
        "10x": tile_qc(tile_points["cell_id"], outlines("cell_boundaries"), outlines("nucleus_boundaries"),
                       region["table"]),
        "InstanSeg": tile_qc(cell_names(tile_points["instanseg_cell"], "instanseg_cells"), outlines("instanseg_cells"),
                             outlines("instanseg_nuclei"), region["instanseg_table"]),
        "Session 5": tile_qc(cell_names(tile_points["session5_cell"], "session5_cells"), outlines("session5_cells"),
                             outlines("instanseg_nuclei"), region["session5_table"]),
    }
    display(pd.DataFrame({name: summary(results) for name, results in qc_tile.items()}).round(2))

In [ ]:
if session5_cells is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for ax, (name, key) in zip(axes, [("10x", "cell_boundaries"), ("InstanSeg", "instanseg_cells"),
                                      ("Session 5", "session5_cells")]):
        show(ZOOM, {name: (outlines(key), "yellow")}, ax=ax, title=name)
    plt.show()

**Questions**

- Your session 5 cells came from a threshold you chose. Which metrics would show cells that a threshold missed?
- Depending on your choices, your cells may capture as many transcripts as 10x's. Is that all good news? Compare
  the cell areas and the border admixture.
- Session 5's cells are grown from nuclei by a fixed distance, like 10x's fallback. How does that show in the
  nucleus–cell IoU?
- If you changed your threshold or expansion in session 5 and saved again, which numbers here would you expect to
  change most? (Try it, if you have time.)

## 8. Exercise: grow the InstanSeg cells

A common fix for tight outlines is to grow each cell outward a little, as 10x does when it has only a nucleus.
`sopa.shapes.expand_radius` grows every outline by a fraction of the average cell radius; with
`no_overlap=True` it then splits any overlaps between neighbours down the middle.

The cell below grows the outlines by half the average radius and adds them to `region` as
`"instanseg_expanded"`.

In [ ]:
expanded = sopa.shapes.expand_radius(region["instanseg_cells"], expand_radius_ratio=0.5, no_overlap=True)
geometry = expanded[["geometry"]].copy()
geometry.attrs.clear()  # Sopa's result still carries the old transforms; we pass them below instead
geometry = geometry[~geometry.is_empty]  # removing overlaps can leave a few cells with nothing; drop them
region["instanseg_expanded"] = ShapesModel.parse(
    geometry,
    transformations=get_transformation(region["instanseg_cells"], get_all=True),
)

show(ZOOM, {"InstanSeg": (outlines("instanseg_cells"), "cyan"),
            "InstanSeg, expanded": (outlines("instanseg_expanded"), "yellow")}, title="Growing the outlines")
plt.show()

**Exercise 2.** Measure the expanded segmentation the same way as before, and add it to the comparison:

1. aggregate the transcripts over `"instanseg_expanded"` into a table called `"instanseg_expanded_table"`;
2. assign the transcripts to the expanded cells (`sopa.spatial.assign_transcript_to_cell`), and convert the
   numbers to names with `cell_names`;
3. run `run_segtraq` and add the result to `qc` as `"InstanSeg, expanded"`. The nuclei don't change.

Then rerun the summary table. What improved, and what got worse?

In [ ]:
sopa.aggregate(
    region,
    shapes_key="instanseg_expanded",
    image_key="morphology_focus",
    points_key="transcripts",
    gene_column="feature_name",
    key_added="instanseg_expanded_table",
    min_transcripts=0,
    min_intensity_ratio=0,
)
sopa.spatial.assign_transcript_to_cell(
    region, points_key="transcripts", shapes_key="instanseg_expanded", key_added="expanded_cell", unassigned_value=0
)
expanded_ids = cell_names(region["transcripts"]["expanded_cell"].compute(), "instanseg_expanded")

qc["InstanSeg, expanded"] = run_segtraq(
    expanded_ids, outlines("instanseg_expanded"), outlines("instanseg_nuclei"), region["instanseg_expanded_table"]
)
pd.DataFrame({name: summary(results) for name, results in qc.items()}).round(2)

**Questions**

- How far would you grow the cells? Try a few values of `expand_radius_ratio`.
- Growing the cells changes the transcript counts but not the cells themselves. Which downstream results would
  change most: cell counts, cell types, or which cells are next to which?

## Saving your cells for session 7

In session 7 you will find cell types in your own InstanSeg cells, as well as in 10x's. The cell below saves the
outlines and their table into the `data` folder, as a small SpatialData Zarr of its own. (If you change anything
above, for example the channels, run it again.)

In [ ]:
SESSION6_CELLS = DATA.parent / "session6_cells.zarr"
saved = sd.SpatialData(shapes={"instanseg_cells": region["instanseg_cells"]},
                       tables={"table": region["instanseg_table"]})
saved.write(SESSION6_CELLS, overwrite=True)
print(f"Saved {region['instanseg_table'].n_obs} cells to {SESSION6_CELLS}")

## Summary

- Segmentation decides which transcripts and which pixels belong to each cell, so its errors carry into every
  later result.
- SpatialData keeps images, transcripts, outlines and tables in one Zarr store, each placed in shared coordinate
  systems by its transformations; objects from QuPath join them as shapes.
- Sopa runs a segmentation model on patches of a large image, and `sopa.aggregate` turns the outlines into a
  cell-by-gene table with protein intensities. Name the table with `key_added` so 10x's isn't overwritten.
- There is no ground truth, but metrics such as unassigned transcripts, transcripts per cell, nucleus–cell
  overlap and border admixture show how two segmentations differ, and in which direction a change moves them.
- Every choice is a trade-off: tight outlines lose transcripts; generous ones borrow them from neighbours.

In session 7 we use the cell tables, 10x's and the ones you just saved, to find cell types, and compare
them with your Day 1 phenotypes.

**Further reading**

- [Sopa documentation](https://prism-oncology.github.io/sopa), including other segmentation methods (Cellpose,
  Baysor, Proseg) that you can swap in at step 3.
- [SegTraQ](https://github.com/LazDaria/segtraq), with the full list of metrics.
- [InstanSeg](https://github.com/instanseg/instanseg).
- [CellSurvey](https://github.com/FrancisCrickInstitute/CellSurvey), the Crick pipeline that runs these steps
  on whole slides.